# SymboLM (v1.2) — Stage 2.5 Concept SFT Syntactic Seeding
**Author & Principal Architect:** Sanish Gyawali  
**AI Systems Collaborator:** Antigravity (Google DeepMind)  
**Repository:** https://github.com/gyawalisanish0/symboLM  

### Objective:
Teach the model the **Register v2.0 cognitive syntax** (`<reg:plan>`, `<reg:deduce>`, `<reg:verify>`, `<reg:bypass>`, `<reg:intent>`) via Supervised Fine-Tuning (SFT) over 1,600 curated competition demonstrations. This eliminates the **Cold-Start Exploration Barrier**, ensuring the policy enters downstream GRPO with non-zero probability over macro-planning tags.

In [ ]:
# Cell 1: Environment & Hardware Verification
import os
import sys
import torch

print(f'Python version: {sys.version}')
print(f'Torch version : {torch.__version__}')
if torch.cuda.is_available():
    print(f'[OK] NVIDIA CUDA detected! Found {torch.cuda.device_count()} GPU(s):')
    for i in range(torch.cuda.device_count()):
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB VRAM)')
else:
    print('[WARNING] CUDA is NOT available! Check accelerator settings.')


In [ ]:
# Cell 2: Install required libraries & resolve package conflicts
!pip install -q --upgrade pip
!pip uninstall -y -q torchao
!pip install -q 'transformers>=4.48.0' 'peft>=0.14.0' 'accelerate>=1.2.0' 'bitsandbytes>=0.45.0' 'datasets>=3.2.0'


In [ ]:
# Cell 3: Clone latest SymboLM repository
!rm -rf /kaggle/working/symboLM
!git clone https://github.com/gyawalisanish0/symboLM.git /kaggle/working/symboLM
%cd /kaggle/working/symboLM


In [ ]:
# Cell 4: Generate the 1,600-sample Unified Stage 3 Curriculum
!python -m data.build_concept_math_dataset
!python -m data.build_stage3_curriculum


In [ ]:
# Cell 5: Launch Stage 2.5 Concept SFT Training on GPU
# Trains LoRA on base model with prompt masking (loss only on reasoning + answer tokens)
!python -m training.sft_train \
    --base_model deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B \
    --train_file data/stage3_unified_curriculum.jsonl \
    --output_dir checkpoints/sft_concept_adapter \
    --epochs 2 \
    --batch_size 8 \
    --grad_accum 2 \
    --lr 2e-4 \
    --no_quant


In [ ]:
# Cell 6: Post-Training Policy Audit (Syntax Emission Verification)
import torch
from inference.generate import load_symbo_model

print('[AUDIT] Loading freshly fine-tuned Concept SFT Adapter...')
model, tokenizer = load_symbo_model('deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B', 'checkpoints/sft_concept_adapter')
model.eval()

test_queries = [
    'If x + 1/x = 5, find the exact value of x^4 + 1/x^4.',
    'Find the smallest positive integer n such that n ≡ 3 (mod 5), n ≡ 5 (mod 7), and n ≡ 7 (mod 11).',
    'What is the atomic number of Carbon?',
    'In a right triangle with integer legs a and b, inradius is r = 6 and perimeter is P = 72. Find area.'
]

for q in test_queries:
    print('\n' + '=' * 60)
    print(f'Problem: {q}')
    prompt = f'<｜begin of sentence｜><｜User｜>{q}<｜Assistant｜><think>\n'
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=192, temperature=0.2, do_sample=True, pad_token_id=tokenizer.eos_token_id)
    gen_text = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=False)
    print('[Generation]')
    print(gen_text)


In [ ]:
# Cell 7: Package and archive final Stage 2.5 policy checkpoints
import os
adapter_cfg = '/kaggle/working/symboLM/checkpoints/sft_concept_adapter/adapter_config.json'
if not os.path.exists(adapter_cfg):
    raise FileNotFoundError(f'CRITICAL: {adapter_cfg} not found! SFT training failed!')

!tar -czf /kaggle/working/symboLM_sft_concept_policy.tar.gz -C /kaggle/working/symboLM/checkpoints/sft_concept_adapter .
print('[COMPLETE] Stage 2.5 Concept SFT complete! Policy weights saved to /kaggle/working/symboLM_sft_concept_policy.tar.gz')
